# __Lecture 08: Finding patterns in biological data__

----
### **_Be sure to pull the latest notebook!!_**

#### __Announcement:__ Homework 3 is due October 27. Homework 4 will be assigned October 29.

#### __Last week, you talked about:__
- Python! (e.g. data types, lists, dictionaries, for loops, if statements, functions)

#### __Today we will:__
- warm up by writing a function that uses a dictionary
- talk about why __searching for patterns__ comes up constantly in computational biology
- learn to __read__ regular expressions using the `re` module
- __write tests__ for pattern-matching code, and use them to catch a bug

-----
# __Warm-up: defining a function__

Everything we will do today builds on the Python from lectures 6 and 7: dictionaries, `for` loops, `if`
statements, and functions with default arguments. 

#### __In groups (5 minutes):__ write a function that uses a dictionary to count how many times each base occurs.

Be sure to account for sequences that contain uppercase and lowercase bases. I've provided an empty
function here (with the docstring already written). Use the test cell below to check your work.

__A note on docstrings:__ There are several styles of docstrings. I'll be writing functions with [Google-style docstrings](https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings),
like the one below: a one-line summary, then `Args:`, then `Returns:`. Writing the docstring
*before* the code is a good way to force yourself to decide what the function is supposed to do.

In [ ]:
def count_bases(seq, case="upper"):
    """Count the number of times each base occurs in a sequence.

    Args:
        seq: DNA sequence, as a string.
        case: Case in which to count the bases, either "upper" or "lower".
            Defaults to "upper".

    Returns:
        A dict keyed by each nucleotide, where the value is the number of times
        that nucleotide is observed in the sequence.

    Note:
        <object>.upper() will convert the <object> string to uppercase.
    """

    assert case in ["lower", "upper"], "case needs to be either lower or upper"

    # YOUR CODE HERE

Notice that the third line below doesn't print anything. Instead, it *stops*, because of
the `assert` statement in the function. We will come back to `assert` in a big way later today.

In [ ]:
print(count_bases("AATXXGGCT"))
print(count_bases("aatTGGcT", case="lower"))

count_bases("ATGCG", case="unicorn")

----
# __Part 1: Why search for patterns?__

As we start working with sequencing data, we will want to be able to search sequences for patterns
of interest. A surprising amount of computational biology is __text parsing__. Here are a few examples you might run into:

In [ ]:
# 1. An influenza strain name downloaded from a sequence database.
#    Q: what year was it collected? What subtype is it?
strain = "A/New York/3/1994 (H3N2)"

# 2. A sequencing read containing a 6-nt barcode (lower case) flanked by a constant sequence on each side.
#    Q: what is the barcode?
barcode = "CTAGCatcgatGATCA"

# 3. FASTQ file names from a shared directory, named by three different people.
#    Q: which gene and which replicate is each file?
filenames = [
    "Sample_01_HA_rep1_R1.fastq.gz",
    "sample-02-NA-rep2_R1.fastq.gz",
    "S03.HA.rep1.R1.fastq.gz",
]

print(strain)
print(barcode)
print(filenames)

Last week, you learned that you can ask whether one string contains another with the `in` keyword,
and that we can break a string apart with `.split()`. Both are useful, but limited.

In [ ]:
# `in` can confirm a subtype we already know about...
print("H3N2" in strain)

# ...but it can't tell us the subtype of a strain we haven't seen before
mystery_strain = "A/swine/Iowa/15/1930 (H1N1)"
print(
    "H3N2" in mystery_strain
)  # False -- but there IS a subtype here, we just don't know it yet

# .split() only helps when the text is perfectly consistent
print("Sample_01_HA_rep1_R1.fastq.gz".split("_"))
print("sample-02-NA-rep2_R1.fastq.gz".split("_"))

# and .split() will happily hand us a "barcode" that could not possibly be one
print(barcode.upper().split("CTAGC")[1].split("GATCA")[0])  # ATCGAT -- correct!
print(
    "CTAGCATYGATGATCA".split("CTAGC")[1].split("GATCA")[0]
)  # ATYGAT -- Y is not a base
print(
    "CTAGCTACAGGATCA".split("CTAGC")[1].split("GATCA")[0]
)  # TACAG -- only 5 bases long

To use `in`, you have to already know the exact text you are looking for. `.split()` will cut a
string wherever you tell it to, but it has no idea what the pieces are supposed to look like - it
cannot tell a six-base barcode from five bases and a typo.

We usually don’t want to find whatever appears at a particular *position* in a string. Instead, we want to find whatever matches a particular *pattern*, regardless of where it appears.

Searching for a pattern involves three steps:

1. __Compile__ a search pattern
2. __Scan__ the text for matches to that pattern
3. __Report__ the matching text

We already have tools for scanning through text and searching for *specific* strings, but we’re missing a way to describe more *general patterns*.

---
# __Part 2: Reading regular expressions__

Python has an extensive collection of built-in __modules__ containing special purpose functions and
objects - we have already used `math`, and on Thursday we will use `biopython`. Today's is `re`.
([The full list of built-in modules is here](https://docs.python.org/3/library/).)

#### A __regular expression__ is a sequence of characters that forms a __search pattern__.

The `re` module compiles that pattern into a matching engine and runs it against your text. The
workflow is almost always the same three steps: __compile__, __search__, __use the result__.

In [ ]:
import re

# This is an example string
seq = "ATTCCGAAA"

# 1. COMPILE the search pattern (note the r before the quotes -- more on that below)
search_pattern = re.compile(r"TC+G")

# 2. SEARCH the sequence with it
match = search_pattern.search(seq)

# 3. USE/REPORT the result. A match object knows WHAT matched and WHERE.
print(match)
print("matched text:", match.group())
print("position:", match.start(), "to", match.end())

#### When there is nothing to find, you get `None` back - not an error

This is common source of confusion with `re`. 

In [ ]:
no_match = search_pattern.search("TTTTTT")

print(no_match)
print(no_match is None)

# this is why you ALWAYS check before calling .group()
# no_match.group()   # <- uncomment to see the error you will definitely hit at some point

__A note on quotes:__ you will see search patterns written as _raw strings_, like `r"\d{4}"`. The `r`
tells Python to leave the backslashes alone instead of interpreting them (`"\n"`, for example, is a
newline rather than a backslash and an `n`). Regular expressions use backslashes constantly, so
__always write your patterns as `r"..."`__ - it costs nothing and avoids a confusing class of bug.

## Pattern cheat sheet

You do __not__ need to memorize this. Keep it open while you read patterns - that is what it is for.

#### Matching specific characters

| pattern | matches |
| --- | --- |
| `A`, `7`, `-` | that exact character |
| `\.` `\$` `\(` `\)` `\*` `\+` | a *literal* `.` `$` `(` `)` `*` `+` - characters that have a special meaning below must be escaped with `\` |

#### Matching a class of characters

| pattern | matches |
| --- | --- |
| `.` | any single character |
| `\d` | any digit, `0`-`9` |
| `\s` | any whitespace character |
| `[ACGTN]` | any one of `A`, `C`, `G`, `T`, `N` (i.e. any one nucleotide) |
| `[a-n]` | any one lower-case letter from `a` to `n` |

#### How many times

| pattern | matches |
| --- | --- |
| `a+` | one or more `a` |
| `a{6}` | exactly six `a` |
| `a{6,8}` | between six and eight `a` |

#### Where in the string

| pattern | matches |
| --- | --- |
| `^A` | an `A` at the *beginning* of the string |
| `A$` | an `A` at the *end* of the string |

#### Grouping and naming

| pattern | meaning |
| --- | --- |
| `(...)` | group this part of the pattern together, and capture what it matched |
| `(?P<name>...)` | capture it *and* give it a name, so you can ask for it by name later |

There are many more special codes (see [the full docs](https://docs.python.org/3/library/re.html)),
and [regex101.com](https://regex101.com) will show you, piece by piece, what a pattern is doing.

## A worked example: reading a pattern one piece at a time

The trick to reading a regular expression is to go left to right and translate each piece into
English, using the tables above. Here is the influenza strain name from Part 1, and a pattern that
pulls the collection year out of it.

In [ ]:
strain = "A/New York/3/1994 (H3N2)"

# r"/(?P<year>\d{4})" reads as:
#   /           a literal forward slash
#   (?P<year>   start a captured group, and call it "year"
#   \d{4}       exactly four digits
#   )           end the group

year_pattern = re.compile(r"/(?P<year>\d{4})")

match = year_pattern.search(strain)

print(match)
print("year:", match.group("year"))

#### __Groups__ and __names__

Grouping and naming allows us to get *part* of a match back out:

* **Grouping**: `(...)` treats everything inside the parentheses as a single unit, and __captures__
  what it matched so you can ask for it separately from the rest of the pattern.
* **Naming**: `(?P<name>...)` captures it *and* labels it, so you can ask for it by name later with
  `match.group("name")` instead of counting parentheses.

Without the group, `year_pattern.search(strain).group()` would give us `"/1994"` - the slash
included - because the whole pattern matched. The group lets us say "match the slash, but only
*give me back* the four digits."

----
### __Practice 1 (in groups, ~10 minutes): read these patterns__

For each cell below:

1. __Before running it__, write down in the comment what shape the pattern describes, in plain
   English, and what you predict will be printed.
2. Then run the cell and compare.

In [ ]:
# (a) What shape does this describe? What will it print?
# your answer:

subtype_pattern = re.compile(r"\((?P<subtype>[A-Z]\d+[A-Z]\d+)\)")

for s in [
    "A/New York/3/1994 (H3N2)",
    "A/swine/Iowa/15/1930 (H1N1)",
    "B/Victoria/2/1987",
]:
    match = subtype_pattern.search(s)
    print(s, "->", match.group("subtype") if match else "no subtype found")

In [ ]:
# (b) Same pattern, with and without the ^ anchor. Which of these three find a match, and why?
# your answer:

loose = re.compile(r"ATG")
anchored = re.compile(r"^ATG")

print(loose.search("GGATGCC"))
print(anchored.search("GGATGCC"))
print(anchored.search("ATGCCGG"))

In [ ]:
# (c) Why does the gene name come out too long?
# your answer:

greedy = re.compile(r"Sample_\d+_(?P<gene>.+)_")

print(greedy.search("Sample_01_HA_rep1_R1.fastq.gz").group("gene"))

----
### __Practice 2 (in groups, ~10 minutes): now write two of your own__

Reading is most of the job, but you should write a couple from scratch so that you can practice with the syntax.
Both of these pull a __barcode__ out of a read, using the same design we have been working with: a
short barcode flanked by a constant sequence on each side. 

__(a)__ A library uses the construct `GGATC`-`NNNNNNNN`-`TCTAG`: an __8-nucleotide barcode__
between the constant sequences `GGATC` and `TCTAG`. Write a pattern that pulls that barcode out,
named `barcode`. Only `A`, `C`, `G`, `T` and `N` should be allowed in it - so the second read below,
whose barcode contains a `Y`, should __not__ match.

__(b)__ Now the same construct, but the sequencing design means the read should contain the
construct and __nothing else__: `GGATC` at the very beginning and `TCTAG` at the very end. Write a
pattern that only matches in that case. You will need anchors.

In [ ]:
# (a) an 8-nt barcode between the constant sequences GGATC and TCTAG
reads = [
    "GGATCACGTACGTTCTAG",  # barcode ACGTACGT
    "GGATCACGTACGYTCTAG",  # barcode contains a Y -- should NOT match
    "GGTTCACGTACGTTCTAG",  # upstream sequence is wrong -- should NOT match
]

bc_pattern = re.compile(r"")  # <-- your pattern here

for read in reads:
    match = bc_pattern.search(read)
    print(read, "->", match.group("barcode") if match else "no match")

In [ ]:
# (b) the same construct, but it must make up the WHOLE read
anchored_reads = [
    "GGATCACGTACGTTCTAG",  # barcode ACGTACGT
    "TTTGGATCACGTACGTTCTAGTTT",  # extra sequence on both sides -- should NOT match
]

anchored_pattern = re.compile(r"")  # <-- your pattern here

for read in anchored_reads:
    match = anchored_pattern.search(read)
    print(read, "->", match.group("barcode") if match else "no match")

----
# __Part 3: Checking that a search pattern is right__

It is easy to write an incorrect regular expression that looks like a correct one. For example, it could:
- match things it should not
- miss things it should catch

Neither failure crashes, and neither prints a warning.

__It is a valuable skill to be able to say what the pattern should do, and then check that it does.__
This is where __*testing*__ comes in!

#### The testing cycle

Testing a piece of code is a loop:

1. __State the requirements__ - what does a correct answer look like?
2. __Write the function/code__
2. __Write test cases__ - one input paired with the answer you expect, one for each requirement
3. __Run the tests__, and run the fucntion/code on real data to find requirements you missed
4. __Fix the code__, and re-run every test

## Step 1: write down the requirements

Before testing anything (and before writing the pattern, ideally) we have to state the design
precisely. Our construct is `CTAGC`-`NNNNNN`-`GATCA`, which means a valid read must satisfy all of:

__Requirements:__
1. the read __starts__ with the fixed upstream sequence `CTAGC` 
2. the barcode is exactly __6__ bases long 
3. every barcode base is one of `A`, `C`, `G`, `T`, `N` - and nothing else 
4. the barcode is followed by the fixed downstream sequence `GATCA` 
5. reads may arrive in lower case, and barcodes are reported in upper case 

Anything that fails one of these is not a valid barcode, and the parser should return `None` for it. We want to test this functionality!

## Step 2: write the search pattern and parser (given to you)

In [ ]:
inherited_pattern = re.compile(r"CTAGC(?P<barcode>[A-Z]{6})GATCA")


def parse_barcode_v1(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = inherited_pattern.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")

## Step 3: write tests

A __test case__ is one input paired with the answer we expect. The `assert` statement is the
simplest way to write one down: `assert <something true>` does nothing at all, and
`assert <something false>` stops with an `AssertionError` (as we saw in the warm-up).

You can put a message after a comma, and that message is what you will see if/when
the test fails:

```python
assert parse_barcode_v1("CTAGCatcgatGATCA") == "ATCGAT", "should accept a good read"
```

Each requirement usually needs two kinds of case:

* a __positive__ case - a good read, and the barcode it should give back
* a __negative__ case - an input that violates __that one requirement__, for which the parser
  should return `None`

One thing to know: a cell stops at the __first__ assert that fails, so any tests after it will not
run. If the cell runs to the end and prints, every test in it passed.

Let's write one of each together:

In [ ]:
# positive: a good read, in lower case (R5)
assert parse_barcode_v1("CTAGCatcgatGATCA") == "ATCGAT", "should accept a good read"

# negative: the upstream sequence is CCAGC, not CTAGC (R1)
assert parse_barcode_v1("CCAGCatagcaGATCA") is None, "R1: upstream sequence is wrong"

# nothing above printed, which means both tests passed
print("all tests passed!")

----
# __In your groups (~25 minutes)__

Work through __Practice 3a__, __3b__, and __3c__ below, in order:

* __3a__ - finish the test set, so that there is a test for every requirement
* __3b__ - run the parser on reads from the actual sequencing run, and find the case your tests missed
* __3c__ - fix the pattern, until every test passes

### __Practice 3a: turn the requirements into test cases__

__Your task:__ write an `assert` statement for each of requirements 1-5 in the cell below.

Work down the requirement list and, for each one, ask: __what input would violate this requirement,
and only this one?__ That input is a __negative__ case, and the parser should return `None` for it.
Include at least one __positive__ case too (a good read, and the barcode it should give back).

The two we wrote together are repeated for you, to show the format. Then run the cell and see how
the inherited parser does.

In [ ]:
# from class -- positive: a good read, in lower case (R5)
assert parse_barcode_v1("CTAGCatcgatGATCA") == "ATCGAT", "should accept a good read"

# from class -- negative: the upstream sequence is CCAGC, not CTAGC (R1)
assert parse_barcode_v1("CCAGCatagcaGATCA") is None, "R1: upstream sequence is wrong"

# YOUR ASSERTS HERE


print("all tests passed!")

### __Practice 3b: run the parser on reads from the actual sequencing run__

Once you've written your tests and they all pass, run the parser on a list of "actual" reads below.
Are all of the results what you expect?

In [ ]:
seqs = [
    "CTAGCatcgatGATCA",
    "CTAGCgaccatGATCA",
    "CTAGCATYGATGATCA",
    "GGGGGCTAGCATCGATGATCATTTT",
    "CTAGCatcgatGATCA",
    "CTAGCtacagGATCA",
    "CCAGCatagcaGATCA",
]

for seq in seqs:
    print(f"{seq:<28} -> {parse_barcode_v1(seq)}")

If any of the reads above came back with a "barcode" that is not a barcode (e.g. doesn't satisfy our requirements above), work out the following:

1. __which requirement__ does that read violate?
2. __why didn't our test set catch it?__ Hint: a single requirement can be violated in more than one way...

__Your task:__ write an `assert` for each missing case in the cell below. You are trying to make
this cell fail.

In [ ]:
# add an assert for each case our requirement list missed
# YOUR ASSERTS HERE


print("all tests passed!")

### __Practice 3c: fix the pattern so that it passes all of the tests__

Now edit the pattern so that every test you have written passes - the ones from 3a __and__ the ones
from 3b. Copy all of your asserts into the bottom of the cell, swapping `parse_barcode_v1` for
`parse_barcode_v2`, so that you are re-running the whole test set against the fixed parser.

In [ ]:
fixed_pattern = re.compile(r"")  # <-- your fixed pattern here


def parse_barcode_v2(seq):
    """Find the barcode in a sequence.

    Args:
        seq: DNA sequence, as a string.

    Returns:
        The barcode as an upper-case string, or None if the sequence does not
        match the expected pattern.
    """
    match = fixed_pattern.search(seq.upper())
    if match is None:
        return None
    return match.group("barcode")


# copy your assert statements from Practice 3a and 3b down here, with parse_barcode_v2
# YOUR ASSERTS HERE


print("all tests passed!")

# __Back together: what did the tests catch?__

1. __Which requirement__ did the inherited pattern violate, and why did `[A-Z]{6}` let those reads through?
2. __Why wasn't one test per requirement enough?__

----
# __Wrapping up__

1. Data arrives as semi-structured text, and we often want to pull some sort of structure out of it.
2. A pattern search is just __scan, test, report__. `in` and `.split()` can scan, but they cannot
   say what a valid match looks like; regular expressions (and the `re` module) can.
3. Being able to __read__ a pattern and translate it into English is a helpful first step
4. Wrong patterns can fail silently, so write test cases (including __negative__ ones)!

#### On Thursday (lecture 09)

We will use the `biopython` module to read a real FASTQ file from an influenza sequencing run

----
# __If extra time: Making the parser reusable__

Our pattern has `CTAGC`, `6`, and `GATCA` baked into it. The next experiment will use a different
construct, and we do not want to rewrite - and re-test - the whole thing then.

### __Practice 4 (in groups, ~15 minutes)__

__Part (a):__ finish `count_barcodes` below. It should take the design as __arguments with default
values__, and return a dictionary of barcode counts. Your function needs to:

1. build the search pattern __from the arguments__, so the same function works for any design
   (`bclen` is an integer, so you will need `str(bclen)` to paste it into the pattern)
2. convert each sequence to uppercase
3. search it, and skip the sequence if there is no match
4. if it matches, extract the barcode and add it to a dictionary of counts

__Part (b):__ before you trust it, __write tests for it__ in the cell after. We know what the right
answer is for `seqs`: `ATCGAT` appears twice, `GACCAT` appears once, and three sequences should be
rejected. Write `assert` statements for those. Then write one more test that checks the
__arguments actually do something__ - for instance, parsing these same sequences with a different
expected `upstream` sequence should find nothing at all.

In [ ]:
def count_barcodes(seqs, bclen=6, upstream="CTAGC", downstream="GATCA"):
    """Parse and count barcodes.

    Args:
        seqs: List of DNA sequences.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode.
        downstream: Sequence expected immediately downstream of the barcode.

    Returns:
        A dict keyed by each valid barcode, where the value is the number of
        times that barcode is observed.

    Note:
        The function is **not** case-sensitive, and all barcodes are reported
        in upper case. The construct must start at the beginning of the sequence.
    """

    # YOUR CODE HERE


print(count_barcodes(seqs))

In [ ]:
# Part (b): write your tests here
counts = count_barcodes(seqs)

# YOUR ASSERTS HERE


print("all checks passed")